# 🎙️ TurboVoiceCloner: Final Stable Build
**System:** Isolated Python 3.11 (cb311) | **Engine:** Chatterbox-v2

### 🚀 निर्देश:
1. **GPU:** सुनिश्चित करें कि T4 GPU चालू है।
2. **Step 1, 2, 3** को क्रम से चलाएं।

In [ ]:
%%bash
# Step 1: Micromamba setup (आपके द्वारा दिया गया कोड)
set -e
cd /content
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
./bin/micromamba create -y -n cb311 -c conda-forge python=3.11 pip
echo "✅ Micromamba ready! Env: cb311 created."

In [ ]:
%%bash
# Step 2: High-Speed Installation (आपके द्वारा दिया गया कोड)
set -euo pipefail
MICROMAMBA="/content/bin/micromamba"

echo "Installing PyTorch 2.5.1 and Dependencies..."
$MICROMAMBA run -n cb311 pip install -U pip setuptools wheel -q
$MICROMAMBA run -n cb311 pip install torch==2.5.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121 -q

echo "Installing Chatterbox-v2 Core..."
$MICROMAMBA run -n cb311 pip install --no-cache-dir --upgrade git+https://github.com/devnen/chatterbox-v2.git@master -q
echo "✅ Installation complete at Turbo Speed!"

In [ ]:
# Step 3: Launch Full Stable Server (आपके द्वारा दिया गया कोड)
import os, subprocess, socket
from google.colab.output import serve_kernel_port_as_window

PORT = 8004
REPO_DIR = "/content/Chatterbox-TTS-Server"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/devnen/Chatterbox-TTS-Server.git"], check=True)

os.chdir(REPO_DIR)
print("🚀 Starting Turbo Server... Model is loading.")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(["/content/bin/micromamba", "run", "-n", "cb311", "python", "-u", "server.py"], 
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)

while True:
    line = proc.stdout.readline()
    if line:
        print(line, end="")
        if "Uvicorn running on" in line:
            print("\n🔥 UI LOADING... CLICK BELOW")
            serve_kernel_port_as_window(PORT)
    if proc.poll() is not None: break